# Implementation 1!

To summarize:
1. Choose your baseline paper
2. Load the dataset(s) and model(s) used (no retraining!!)
3. MOST IMPORTANT!!! Understand their quantization pipeline and how they do bit budgeting
4. Evaluate the sensitivity metric proposed
5. (BONUS!) Evaluate another sensitivity metric from another domain or come up with one!

In [ ]:
%pip install -q torch transformers datasets accelerate sentencepiece

# Imports & Constants

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

SEED = 42

MODEL_ID = "facebook/opt-125m"

CALIB_DATASET = "mit-han-lab/pile-val-backup"
EVAL_DATASET = "Salesforce/wikitext"
EVAL_CONFIG = "wikitext-2-raw-v1"

N_CALIB_SAMPLES = 128
CALIB_SEQ_LEN = 512

torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
CUDA available: True


# Define and load Dataset(s) (plural if in the paper 🙂)

AWQ uses a small calibration set from the Pile to collect activation statistics from the model. This calibration data is not used to retrain the model.

For evaluation, I am also using WikiText-2, which the AWQ paper uses to measure perplexity before and after quantization.

In [ ]:
pile_calib = load_dataset(
    CALIB_DATASET,
    split="validation"
)

pile_calib = pile_calib.shuffle(seed=SEED)

print("Pile loaded successfully!")
print("Number of examples:", len(pile_calib))

print("\nExample text:")
print(pile_calib[0]["text"][:500])

Repo card metadata block was not found. Setting CardData to empty.


Generating validation split:   0%|          | 0/214670 [00:00<?, ? examples/s]

Pile loaded successfully!
Number of examples: 214670

Example text:
Estrogens interact in a variety of ways with the liver. Not only is the liver an estrogen-responsive tissue, but it is also responsible for the interconversion, metabolism and excretion of the estrogens. The effect of liver disease on these processes has not been studied well, except in the specific case of alcohol-induced liver disease. Therefore, using five well-characterized animal models of liver diseases, we will assess the effect of a variety of liver diseases on several aspects of estroge


In [ ]:
wikitext_test = load_dataset(
    EVAL_DATASET,
    EVAL_CONFIG,
    split="test"
)

print("WikiText-2 loaded successfully!")
print("Number of examples:", len(wikitext_test))

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

WikiText-2 loaded successfully!
Number of examples: 4358


# Define and load the model(s) you are using (plural if in the paper 🙂)
load pretrained/weights (you could probably get the weights not even load the model itself)

I am using OPT-125M for the initial AWQ implementation. The AWQ paper evaluates the OPT model family, and I am starting with the smaller version so I can test the full workflow with limited compute before scaling up.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=False
)

print("Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16
).to("cuda")

model.eval()

num_params = sum(p.numel() for p in model.parameters())

print("Model loaded successfully!")
print("Model:", MODEL_ID)
print(f"Parameters: {num_params / 1e6:.2f} million")
print("Device:", next(model.parameters()).device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  251MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  251MB            

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model.safetensors: downloading bytes:           |  0.00B            

Model loaded successfully!
Model: facebook/opt-125m
Parameters: 125.24 million
Device: cuda:0


In [ ]:
test_text = "Machine learning models can"

inputs = tokenizer(
    test_text,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model(**inputs)

print("Input shape:", inputs["input_ids"].shape)
print("Output shape:", outputs.logits.shape)
print("Forward pass successful!")

Input shape: torch.Size([1, 5])
Output shape: torch.Size([1, 5, 50272])
Forward pass successful!


# Define the function for the sensitivity metric(s) used
This includes both the paper's proposed sensitivity metric and the paper's baselines that they compare their metric to. You probably need to put more functions defining the paper's baseline sensitivity metrics (yay more githubs), but you can do a simplified version of it (prefer not tho...if you do I will be sad).

Describe the paper's sensitivity metric HERE:
What were the baselines the paper compared their metric to?

AWQ uses activation magnitude to identify salient weight channels. The idea is that channels receiving larger input activations are more important to the model, so they should be protected more carefully during quantization.

In [ ]:
import torch.nn as nn

def collect_activation_stats(model, tokenizer, texts, max_samples=16):
    activation_stats = {}
    hooks = []

    def make_hook(name):
        def hook(module, inputs, output):
            x = inputs[0].detach()

            # average absolute activation over all dimensions except channel
            dims = tuple(range(x.dim() - 1))
            channel_mag = x.abs().mean(dim=dims).cpu()

            if name not in activation_stats:
                activation_stats[name] = {
                    "sum": channel_mag,
                    "count": 1
                }
            else:
                activation_stats[name]["sum"] += channel_mag
                activation_stats[name]["count"] += 1

        return hook

    # attach hooks to linear layers
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            hooks.append(
                module.register_forward_hook(make_hook(name))
            )

    model.eval()

    with torch.no_grad():
        for text in texts[:max_samples]:
            if not text.strip():
                continue

            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to("cuda")

            model(**inputs)

    for hook in hooks:
        hook.remove()

    # convert sums to averages
    final_stats = {}

    for name, info in activation_stats.items():
        final_stats[name] = info["sum"] / info["count"]

    return final_stats

In [ ]:
calib_texts = [
    pile_calib[i]["text"]
    for i in range(32)
    if pile_calib[i]["text"].strip()
]

activation_stats = collect_activation_stats(
    model,
    tokenizer,
    calib_texts,
    max_samples=16
)

print("Collected activation stats for", len(activation_stats), "linear layers")

for name, stats in list(activation_stats.items())[:5]:
    print(
        name,
        "shape:",
        stats.shape,
        "mean activation:",
        stats.mean().item()
    )

Collected activation stats for 73 linear layers
model.decoder.layers.0.self_attn.q_proj shape: torch.Size([768]) mean activation: 0.218505859375
model.decoder.layers.0.self_attn.k_proj shape: torch.Size([768]) mean activation: 0.218505859375
model.decoder.layers.0.self_attn.v_proj shape: torch.Size([768]) mean activation: 0.218505859375
model.decoder.layers.0.self_attn.out_proj shape: torch.Size([768]) mean activation: 0.0282440185546875
model.decoder.layers.0.fc1 shape: torch.Size([768]) mean activation: 0.10870361328125


# !! MOST IMPORTANT !!
## Define the necessary functions that they use to quantize the model and perform bit budgeting

If they use another pipeline from another github, note the link & feel free to add more code cells as needed!

Describe how the paper does bit budgeting:

AWQ does not use layer-wise mixed-precision bit budgeting. Instead, it uses a uniform low-bit setting, mainly INT3 or INT4, with group size 128. It uses a small calibration set to collect activation statistics and identify important (salient) channels. Rather than keeping those channels in FP16, the final AWQ method searches for per-channel scaling factors that protect them while keeping the model at the same low-bit precision. For RQNKUP, AWQ is therefore our strong uniform LLM baseline rather than the mixed-precision bit-allocation method.

Implementation note: The original MIT AWQ repository had dependency conflicts with the current Google Colab environment, so I adapted the baseline using the current Transformers + GPTQModel AWQ loading path. I loaded a Llama-2-7B AWQ checkpoint using 4-bit weights and group size 128.

In [ ]:
# Clone and install the official AWQ implementation
import os

if not os.path.exists("/content/llm-awq"):
    !git clone https://github.com/mit-han-lab/llm-awq.git /content/llm-awq

%cd /content/llm-awq
!pip install -e .

Cloning into '/content/llm-awq'...
remote: Enumerating objects: 1144, done.
remote: Counting objects: 100% (621/621), done.
remote: Compressing objects: 100% (327/327), done.
remote: Total 1144 (delta 455), reused 294 (delta 294), pack-reused 523 (from 2)
Receiving objects: 100% (1144/1144), 183.13 MiB | 21.97 MiB/s, done.
Resolving deltas: 100% (606/606), done.
Updating files: 100% (180/180), done.
/content/llm-awq
Obtaining file:///content/llm-awq
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of awq to determine which version is compatible with other requirements. This could take a while.
ERROR: Could not find a version that satisfies the requirement torch==2.3.0 (from awq) (from versions: 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0,

In [ ]:
# Install AWQ without forcing its old PyTorch version
%cd /content/llm-awq

!pip install -e . --no-deps

# Install the dependencies we actually need for the AWQ baseline
!pip install -q \
    transformers==4.46.0 \
    accelerate==0.34.2 \
    lm_eval==0.3.0 \
    texttable \
    toml \
    attributedict \
    sentencepiece \
    protobuf

/content/llm-awq
Obtaining file:///content/llm-awq
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for awq (pyproject.toml) ... done
  Created wheel for awq: filename=awq-0.1.0-0.editable-py3-none-any.whl size=10337 sha256=f0cac90222f185c23c584b436ea0200d0d93a77982342e3976ffde618755a89a
  Stored in directory: /tmp/pip-ephem-wheel-cache-9b92xh2k/wheels/62/76/e5/e62ea9841aaf055718a60f09cd42dc862ddfc35b3fa935f6d2
Successfully built awq
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import sys
import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

try:
    import awq
    from awq.quantize.pre_quant import apply_awq
    from awq.quantize.quantizer import pseudo_quantize_model_weight
    print("Core AWQ imports: SUCCESS")
except Exception as e:
    print("Core AWQ imports: FAILED")
    print(type(e).__name__, e)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
Transformers: 4.46.0
CUDA available: True
Core AWQ imports: FAILED
ModuleNotFoundError No module named 'awq_inference_engine'


In [ ]:
!pip install -q -U transformers accelerate datasets optimum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 55.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awq 0.1.0 requires accelerate==0.34.2, but you have accelerate 1.15.0 which is incompatible.
awq 0.1.0 requires gradio==3.35.2, but you have gradio 6.29.0 which is incompatible.
awq 0.1.0 requires gradio_client==0.2.9, but you have gradio-client 2.7.1 which is inc

In [ ]:
!pip install -q -U huggingface_hub datasets transformers accelerate optimum

import torch
import transformers
import accelerate
import datasets
import optimum

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA:", torch.cuda.is_available())

Error importing huggingface_hub.hf_file_system: cannot import name '_validate_relative_filename' from 'huggingface_hub._local_folder' (/usr/local/lib/python3.13/dist-packages/huggingface_hub/_local_folder.py)


ImportError: cannot import name '_validate_relative_filename' from 'huggingface_hub._local_folder' (/usr/local/lib/python3.13/dist-packages/huggingface_hub/_local_folder.py)

In [2]:
%cd /content

import torch
import transformers
import datasets
import huggingface_hub
import accelerate
import optimum
from importlib.metadata import version

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("Accelerate:", accelerate.__version__)
print("Optimum:", version("optimum"))
print("CUDA available:", torch.cuda.is_available())

/content
PyTorch: 2.11.0+cu130
Transformers: 5.18.0
Datasets: 5.1.0
HF Hub: 1.33.0
Accelerate: 1.15.0
Optimum: 2.3.0
CUDA available: True


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TheBloke/Llama-2-7B-AWQ"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto"
)

print("Model loaded successfully")
print("Device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

ImportError: Loading an AWQ quantized model requires gptqmodel. Please install it with `pip install gptqmodel`

In [4]:
%pip install -q gptqmodel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 12.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
    

In [1]:
import torch
import transformers
import gptqmodel

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPTQModel imported successfully")
print("CUDA available:", torch.cuda.is_available())


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.5.0
Transformers : 5.18.0
Torch        : 2.11.0+cu130
Triton       : 3.6.0
PyTorch: 2.11.0+cu130
Transformers: 5.18.0
GPTQModel imported successfully
CUDA availabl

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TheBloke/Llama-2-7B-AWQ"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype="auto"
)

print("Model loaded successfully")
print("Device:", next(model.parameters()).device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.89GB            

model.safetensors: downloading bytes:           |  0.00B            

INFO  ExLlamaV2 AWQ: compiling torch.ops JIT extension in `/root/.cache/gptqmodel/torch_extensions/exllamav2_awq/28102340d90d94d7`.
INFO  ExLlamaV2 AWQ: torch.ops JIT extension ready in 57s (estimated ~35s, +22s).
INFO  Kernel: Auto-selection: adding candidate `AwqExllamaV2Linear`             
INFO  Kernel: selected -> `AwqExllamaV2Linear`.                                 


Loading weights:   0%|          | 0/739 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

INFO  gc.collect() reclaimed 6497 objects in 0.353s                             
Model loaded successfully
Device: cuda:0


# Evaluate the model
This includes the paper's own baselines versus the proposed sensitivity metric.

Explain the metrics that they used to evaluate the model on. If you came up with other metrics (bonus points!!), explain them as well:

In [3]:
from datasets import load_dataset
from tqdm.auto import tqdm
import torch
import math

# Show the quantization settings of the model we loaded
print("Quantization config:")
print(getattr(model.config, "quantization_config", "Not found"))

# Load WikiText-2 test data
wikitext = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="test"
)

# Join the non-empty text samples into one long sequence
text = "\n\n".join(
    row["text"] for row in wikitext
    if row["text"].strip()
)

# Tokenize the full evaluation text
encodings = tokenizer(
    text,
    return_tensors="pt",
    add_special_tokens=False
)

input_ids = encodings["input_ids"]

device = next(model.parameters()).device
SEQ_LEN = 2048

total_nll = 0.0
total_tokens = 0

model.eval()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

with torch.inference_mode():
    for start in tqdm(range(0, input_ids.size(1) - 1, SEQ_LEN)):
        end = min(start + SEQ_LEN, input_ids.size(1))

        batch = input_ids[:, start:end].to(device)

        if batch.size(1) < 2:
            continue

        outputs = model(
            batch,
            labels=batch,
            use_cache=False
        )

        predicted_tokens = batch.size(1) - 1

        total_nll += outputs.loss.float().item() * predicted_tokens
        total_tokens += predicted_tokens

ppl = math.exp(total_nll / total_tokens)

print("\n--- AWQ Evaluation ---")
print(f"Model: {MODEL_ID}")
print(f"WikiText-2 Perplexity: {ppl:.3f}")
print("Paper reference (Llama-2-7B FP16): 5.47")
print("Paper reference (Llama-2-7B AWQ INT4-g128): 5.60")

Quantization config:
AwqConfig(quant_method=<QuantizationMethod.AWQ: 'awq'>)


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

  0%|          | 0/166 [00:00<?, ?it/s]


--- AWQ Evaluation ---
Model: TheBloke/Llama-2-7B-AWQ
WikiText-2 Perplexity: 5.628
Paper reference (Llama-2-7B FP16): 5.47
Paper reference (Llama-2-7B AWQ INT4-g128): 5.60


### Evaluation Results

I evaluated the loaded Llama-2-7B AWQ model on the WikiText-2 test set using perplexity (PPL), where lower is better.

- My AWQ INT4 result: **5.628 PPL**
- Published AWQ INT4-g128 result: **5.60 PPL**
- Published FP16 reference: **5.47 PPL**

My result is very close to the published AWQ result, with only about a 0.5% difference. This shows that 4-bit AWQ preserves most of the original model quality while using much lower-precision weights.

This is not an exact reproduction of the authors' original environment, since I used a pre-quantized AWQ checkpoint and a modern Transformers/GPTQModel backend, but the resulting perplexity closely matches the paper.


# More results
Feel free to put as many visuals as you want, the more the better! Remember to not only use the metrics that the paper has but potentially think of more teehee

Explain why you chose the visuals:
If you came up with new metrics (bonus points!), explain them:

In [5]:
# Recover the ORIGINAL model parameter count from the architecture.
# We cannot use sum(p.numel()) on the AWQ model because its weights are packed.

cfg = model.config

hidden = cfg.hidden_size
intermediate = cfg.intermediate_size
layers = cfg.num_hidden_layers
vocab = cfg.vocab_size
heads = cfg.num_attention_heads
kv_heads = getattr(cfg, "num_key_value_heads", heads)

# Dimension used by K/V projections
kv_dim = hidden * kv_heads // heads

# Attention:
# Q = hidden x hidden
# K = hidden x kv_dim
# V = hidden x kv_dim
# O = hidden x hidden
attention_params = (
    hidden * hidden +
    hidden * kv_dim +
    hidden * kv_dim +
    hidden * hidden
)

# Llama MLP has gate_proj, up_proj, down_proj
mlp_params = 3 * hidden * intermediate

# Two RMSNorms per transformer layer
norm_params_per_layer = 2 * hidden

# All transformer layers
transformer_params = layers * (
    attention_params +
    mlp_params +
    norm_params_per_layer
)

# Token embeddings
embedding_params = vocab * hidden

# Llama-2 does not tie lm_head to embeddings
lm_head_params = 0 if getattr(cfg, "tie_word_embeddings", False) else vocab * hidden

# Final RMSNorm
final_norm_params = hidden

num_params = (
    transformer_params +
    embedding_params +
    lm_head_params +
    final_norm_params
)

print(f"Estimated original parameters: {num_params / 1e9:.3f} billion")

# Theoretical weight storage
fp16_bytes = num_params * 2       # 16 bits = 2 bytes
int4_bytes = num_params * 0.5     # 4 bits = 0.5 bytes

fp16_gib = fp16_bytes / (1024**3)
int4_gib = int4_bytes / (1024**3)

print(f"Approx FP16 weight memory: {fp16_gib:.2f} GiB")
print(f"Ideal INT4 weight memory: {int4_gib:.2f} GiB")
print(f"Ideal weight-memory reduction: {(1-int4_gib/fp16_gib)*100:.1f}%")

# Rough dense-model compute proxy per generated token
# This is an approximation, not measured runtime FLOPs.
flops_per_token = 2 * num_params

# Simple precision-aware BOP proxy
fp16_bops = num_params * 16 * 16
awq_bops = num_params * 4 * 16

print(f"\nApprox dense FLOPs/token: {flops_per_token / 1e9:.2f} GFLOPs")
print(f"FP16 BOP proxy/token: {fp16_bops / 1e12:.2f} trillion bit-ops")
print(f"AWQ W4A16 BOP proxy/token: {awq_bops / 1e12:.2f} trillion bit-ops")
print(f"BOP proxy reduction: {(1-awq_bops/fp16_bops)*100:.1f}%")

Estimated original parameters: 6.738 billion
Approx FP16 weight memory: 12.55 GiB
Ideal INT4 weight memory: 3.14 GiB
Ideal weight-memory reduction: 75.0%

Approx dense FLOPs/token: 13.48 GFLOPs
FP16 BOP proxy/token: 1.73 trillion bit-ops
AWQ W4A16 BOP proxy/token: 0.43 trillion bit-ops
BOP proxy reduction: 75.0%


### Deployment Results

For Llama-2-7B, the original model contains approximately **6.74 billion parameters**. Storing those weights in FP16 requires about **12.55 GiB** of raw weight memory, while ideal 4-bit storage requires about **3.14 GiB**, giving a theoretical **75% reduction in weight memory**.

Using a simple dense-model compute approximation, the model requires about **13.48 GFLOPs per generated token**. Quantization does not necessarily reduce the number of mathematical operations, but it lowers the precision of the weight operations. Using a simple bit-operation proxy, W4A16 AWQ reduces the estimated bit-operation cost from **1.73 trillion to 0.43 trillion bit-ops per token**, a theoretical reduction of about **75%**.

Combined with the WikiText-2 result of **5.628 perplexity**, compared with the paper's **5.60 AWQ INT4-g128 result**, this shows that AWQ can substantially reduce model storage and precision while preserving most of the original language-model quality.

# BONUS BONUS BONUS!!!!
Evaluate another sensitivity metric (could be from another paper discussed on monday or a different model). Explain why it succeeded or failed compared to uniform quantization